In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path = r"C:\Users\komal\Videos\Power BI\Data Analyst Interview Test - Answer\Data Analyst Interview Test - Answer\Sales Power BI Assessment\2. Dashboard Data.xlsx"

excel_file = pd.ExcelFile(file_path)

In [4]:
excel_file.sheet_names

['Instruction', 'Sheet1', 'Sheet2', 'Sheet3', 'Sheet4', 'Sheet5', 'Sheet6']

In [5]:
data_sheets = [
    "Sheet1",
    "Sheet2",
    "Sheet3",
    "Sheet4",
    "Sheet5",
    "Sheet6"
]

In [6]:
df1 = pd.read_excel(
    file_path,
    sheet_name="Sheet1",
    header=None
)

df1.head(15)

,0,1,2,3,4,5
0,NaN,NaN,NaN,NaN,NaN,NaN
1,Bill No,Bill Date,Product Name,R/I,Qty,Rate
2,A.S.EMBROIDERY,NaN,NaN,NaN,NaN,NaN
3,VISCOSE 1200MTR,NaN,NaN,NaN,NaN,NaN
4,4753,2022-05-30 00:00:00,6022,I,2,445
5,4927,2022-06-02 00:00:00,6022,I,1,445
6,5496,2022-06-15 00:00:00,500,I,2,445
7,5540,2022-06-16 00:00:00,500,I,2,445
8,5642,2022-06-18 00:00:00,500,I,2,445
9,NaN,NaN,Total,NaN,9,NaN


In [7]:
df1 = df1.iloc[:, :6]

df1.columns = [
    "Bill No",
    "Bill Date",
    "Product Name",
    "R/I",
    "Qty",
    "Rate"
]

df1.head(10)

,Bill No,Bill Date,Product Name,R/I,Qty,Rate
0,NaN,NaN,NaN,NaN,NaN,NaN
1,Bill No,Bill Date,Product Name,R/I,Qty,Rate
2,A.S.EMBROIDERY,NaN,NaN,NaN,NaN,NaN
3,VISCOSE 1200MTR,NaN,NaN,NaN,NaN,NaN
4,4753,2022-05-30 00:00:00,6022,I,2,445
5,4927,2022-06-02 00:00:00,6022,I,1,445
6,5496,2022-06-15 00:00:00,500,I,2,445
7,5540,2022-06-16 00:00:00,500,I,2,445
8,5642,2022-06-18 00:00:00,500,I,2,445
9,NaN,NaN,Total,NaN,9,NaN


In [8]:
#Extra spaces remove karo
for col in df1.columns:
    df1[col] = df1[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

In [10]:
#Header/section rows identify karo
section_mask = (
    df1["Bill No"].notna()
    &
    df1[
        ["Bill Date", "Product Name", "R/I", "Qty", "Rate"]
    ].isna().all(axis=1)
)
df1.loc[section_mask, "Bill No"].head(20)

2         A.S.EMBROIDERY
3        VISCOSE 1200MTR
11       VISCOSE 3000MTR
36      AAGAM ENTERPRISE
40       VISCOSE 1200MTR
70           YELLOW JARI
78        AARCHI FASHION
82        GERMAN 1200MTR
137       GERMAN 1800MTR
144      VISCOSE 1200MTR
330      VISCOSE 3000MTR
345           WHITE JARI
373          YELLOW JARI
378    AASTHA ENTERPRISE
382           CHIKU JARI
388       GERMAN 1200MTR
435      VISCOSE 1200MTR
463      VISCOSE 3000MTR
469           WHITE JARI
474          YELLOW JARI
Name: Bill No, dtype: object

In [12]:
#Actual transaction rows identify karo
date_check = pd.to_datetime(
    df1["Bill Date"],
    errors="coerce"
)

transaction_mask = date_check.notna()
df1.loc[transaction_mask].head(10)

C:\Users\komal\AppData\Local\Temp\ipykernel_21116\3761753059.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_check = pd.to_datetime(


,Bill No,Bill Date,Product Name,R/I,Qty,Rate
4,4753,2022-05-30 00:00:00,6022,I,2,445
5,4927,2022-06-02 00:00:00,6022,I,1,445
6,5496,2022-06-15 00:00:00,500,I,2,445
7,5540,2022-06-16 00:00:00,500,I,2,445
8,5642,2022-06-18 00:00:00,500,I,2,445
12,85,2022-04-01 00:00:00,J500 3000MTR,I,30,64
13,202,2022-04-02 00:00:00,J500 3000MTR,I,45,66
14,442,2022-04-06 00:00:00,J500 3000MTR,I,45,64
15,536,2022-04-07 00:00:00,J500 3000MTR,I,90,64
16,856,2022-04-11 00:00:00,J500 3000MTR,I,75,64


In [13]:
#Customer aur Category ko har row mein fill karna
#Next non-empty row identify karenge
non_empty_rows = [
    i for i in df1.index
    if df1.loc[i].notna().any()
]

In [14]:
#Position mapping
row_position = {
    index: position
    for position, index in enumerate(non_empty_rows)
}

In [15]:
#Customer/Category extraction
current_customer = None
current_category = None

clean_rows = []

for i in df1.index:

    # Section header
    if section_mask.loc[i]:

        position = row_position[i]

        if position + 1 < len(non_empty_rows):

            next_row = non_empty_rows[position + 1]

            # Agar next non-empty row bhi section header hai,
            # current row = Customer Name
            if section_mask.loc[next_row]:
                current_customer = df1.loc[i, "Bill No"]

            # Otherwise current row = Product Category
            else:
                current_category = df1.loc[i, "Bill No"]

    # Actual transaction row
    elif transaction_mask.loc[i]:

        clean_rows.append([
            current_customer,
            current_category,
            df1.loc[i, "Bill No"],
            df1.loc[i, "Bill Date"],
            df1.loc[i, "Product Name"],
            df1.loc[i, "R/I"],
            df1.loc[i, "Qty"],
            df1.loc[i, "Rate"]
        ])

In [17]:
#Clean Sheet1 DataFrame banao
clean_df1 = pd.DataFrame(
    clean_rows,
    columns=[
        "Customer Name",
        "Product Category",
        "Bill No",
        "Bill Date",
        "Product Name",
        "R/I",
        "Qty",
        "Rate"
    ]
)
clean_df1.head(10)

,Customer Name,Product Category,Bill No,Bill Date,Product Name,R/I,Qty,Rate
0,A.S.EMBROIDERY,VISCOSE 1200MTR,4753,2022-05-30,6022,I,2.0,445.0
1,A.S.EMBROIDERY,VISCOSE 1200MTR,4927,2022-06-02,6022,I,1.0,445.0
2,A.S.EMBROIDERY,VISCOSE 1200MTR,5496,2022-06-15,500,I,2.0,445.0
3,A.S.EMBROIDERY,VISCOSE 1200MTR,5540,2022-06-16,500,I,2.0,445.0
4,A.S.EMBROIDERY,VISCOSE 1200MTR,5642,2022-06-18,500,I,2.0,445.0
5,A.S.EMBROIDERY,VISCOSE 3000MTR,85,2022-04-01,J500 3000MTR,I,30.0,64.0
6,A.S.EMBROIDERY,VISCOSE 3000MTR,202,2022-04-02,J500 3000MTR,I,45.0,66.0
7,A.S.EMBROIDERY,VISCOSE 3000MTR,442,2022-04-06,J500 3000MTR,I,45.0,64.0
8,A.S.EMBROIDERY,VISCOSE 3000MTR,536,2022-04-07,J500 3000MTR,I,90.0,64.0
9,A.S.EMBROIDERY,VISCOSE 3000MTR,856,2022-04-11,J500 3000MTR,I,75.0,64.0


In [18]:
#Cleaning function
def clean_sales_sheet(file_path, sheet_name):

    # -----------------------------------
    # 1. Read raw sheet
    # -----------------------------------
    df = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=None
    )

    # Only first 6 columns
    df = df.iloc[:, :6].copy()

    # Standard column names
    df.columns = [
        "Bill No",
        "Bill Date",
        "Product Name",
        "R/I",
        "Qty",
        "Rate"
    ]

    # -----------------------------------
    # 2. Remove extra spaces
    # -----------------------------------
    for col in df.columns:
        df[col] = df[col].apply(
            lambda x: x.strip() if isinstance(x, str) else x
        )

    # -----------------------------------
    # 3. Identify section rows
    # -----------------------------------
    section_mask = (
        df["Bill No"].notna()
        &
        df[
            ["Bill Date", "Product Name", "R/I", "Qty", "Rate"]
        ].isna().all(axis=1)
    )

    # -----------------------------------
    # 4. Identify actual transaction rows
    # -----------------------------------
    parsed_dates = pd.to_datetime(
        df["Bill Date"],
        errors="coerce"
    )

    transaction_mask = parsed_dates.notna()

    # -----------------------------------
    # 5. Get non-empty row positions
    # -----------------------------------
    non_empty_rows = [
        i for i in df.index
        if df.loc[i].notna().any()
    ]

    row_position = {
        index: position
        for position, index in enumerate(non_empty_rows)
    }

    # -----------------------------------
    # 6. Extract Customer / Category
    # -----------------------------------
    current_customer = None
    current_category = None

    clean_rows = []

    for i in df.index:

        # Section header
        if section_mask.loc[i]:

            position = row_position[i]

            if position + 1 < len(non_empty_rows):

                next_row = non_empty_rows[position + 1]

                # Customer Name
                if section_mask.loc[next_row]:
                    current_customer = df.loc[i, "Bill No"]

                # Product Category
                else:
                    current_category = df.loc[i, "Bill No"]

        # Transaction
        elif transaction_mask.loc[i]:

            clean_rows.append([
                current_customer,
                current_category,
                df.loc[i, "Bill No"],
                df.loc[i, "Bill Date"],
                df.loc[i, "Product Name"],
                df.loc[i, "R/I"],
                df.loc[i, "Qty"],
                df.loc[i, "Rate"]
            ])

    # -----------------------------------
    # 7. Create clean DataFrame
    # -----------------------------------
    clean_df = pd.DataFrame(
        clean_rows,
        columns=[
            "Customer Name",
            "Product Category",
            "Bill No",
            "Bill Date",
            "Product Name",
            "R/I",
            "Qty",
            "Rate"
        ]
    )

    return clean_df

In [20]:
#Sheet1 test karo
clean_df1 = clean_sales_sheet(
    file_path,
    "Sheet1"
)
clean_df1.shape

C:\Users\komal\AppData\Local\Temp\ipykernel_21116\2582286349.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(


(13810, 8)

In [22]:
#Customer/category missing check:
clean_df1[
    ["Customer Name", "Product Category"]
].isna().sum()

Customer Name       0
Product Category    0
dtype: int64

In [24]:
#Ab Sheet 1–6 sab clean karo
cleaned_sheets = {}

for sheet in data_sheets:

    cleaned_sheets[sheet] = clean_sales_sheet(
        file_path,
        sheet
    )

    print(
        sheet,
        "Rows:",
        len(cleaned_sheets[sheet])
    )

C:\Users\komal\AppData\Local\Temp\ipykernel_21116\2582286349.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(


Sheet1 Rows: 13810


C:\Users\komal\AppData\Local\Temp\ipykernel_21116\2582286349.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(


Sheet2 Rows: 13245


C:\Users\komal\AppData\Local\Temp\ipykernel_21116\2582286349.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(


...

Sheet4 Rows: 49691


C:\Users\komal\AppData\Local\Temp\ipykernel_21116\2582286349.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(


Sheet5 Rows: 28947


C:\Users\komal\AppData\Local\Temp\ipykernel_21116\2582286349.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(


Sheet6 Rows: 48801


In [25]:
#Har sheet mein missing values check karo
for sheet, df in cleaned_sheets.items():

    print("\n", sheet)

    print(
        df.isna().sum()
    )


 Sheet1
Customer Name       0
Product Category    0
Bill No             0
Bill Date           0
Product Name        0
R/I                 0
Qty                 0
Rate                2
dtype: int64

 Sheet2
Customer Name       0
Product Category    0
Bill No             0
Bill Date           0
Product Name        0
R/I                 0
Qty                 0
Rate                0
dtype: int64

 Sheet3
Customer Name       0
Product Category    0
Bill No             0
Bill Date           0
Product Name        0
R/I                 0
Qty                 0
Rate                0
dtype: int64

 Sheet4
Customer Name          0
Product Category       0
Bill No               18
Bill Date              0
Product Name           0
R/I                    0
Qty                    0
Rate                1358
dtype: int64

 Sheet5
Customer Name       0
Product Category    0
Bill No             0
Bill Date           0
Product Name        0
R/I                 0
Qty                 0
Rate                1

In [26]:
#Data Types Correct Karna
for sheet, df in cleaned_sheets.items():

    # Date
    df["Bill Date"] = pd.to_datetime(
        df["Bill Date"],
        errors="coerce"
    )

    # Numeric columns
    df["Qty"] = pd.to_numeric(
        df["Qty"],
        errors="coerce"
    )

    df["Rate"] = pd.to_numeric(
        df["Rate"],
        errors="coerce"
    )

    # Text columns
    for col in [
        "Customer Name",
        "Product Category",
        "Bill No",
        "Product Name",
        "R/I"
    ]:
        df[col] = df[col].apply(
            lambda x: x.strip() if isinstance(x, str) else x
        )

In [27]:
#Blank Customer/Category check
for sheet, df in cleaned_sheets.items():

    print(
        sheet,
        "\nCustomer missing:",
        df["Customer Name"].isna().sum(),
        "\nCategory missing:",
        df["Product Category"].isna().sum()
    )

Sheet1 
Customer missing: 0 
Category missing: 0
Sheet2 
Customer missing: 0 
Category missing: 0
Sheet3 
Customer missing: 0 
Category missing: 0
Sheet4 
Customer missing: 0 
Category missing: 0
Sheet5 
Customer missing: 0 
Category missing: 0
Sheet6 
Customer missing: 0 
Category missing: 0


In [28]:
#Total rows ka kya karna hai?
transaction_mask = parsed_dates.notna()

NameError: name 'parsed_dates' is not defined

In [29]:
for sheet, df in cleaned_sheets.items():

    total_rows = df[
        df["Product Name"].astype(str).str.strip().str.lower()
        == "total"
    ]

    print(sheet, len(total_rows))

Sheet1 0
Sheet2 0
Sheet3 0
Sheet4 0
Sheet5 0
Sheet6 0


In [30]:
#Negative Qty ko TOUCH NAHI karna
for sheet, df in cleaned_sheets.items():

    print(
        sheet,
        "Negative Qty:",
        (df["Qty"] < 0).sum()
    )

Sheet1 Negative Qty: 2217
Sheet2 Negative Qty: 2320
Sheet3 Negative Qty: 4132
Sheet4 Negative Qty: 6488
Sheet5 Negative Qty: 4293
Sheet6 Negative Qty: 8293


In [32]:
#Ab 6 cleaned sheets ko APPEND karo
master_df = pd.concat(
    cleaned_sheets.values(),
    ignore_index=True
)
master_df.shape

(185689, 8)

In [33]:
#Duplicate check
master_df.duplicated().sum()

np.int64(10001)

In [34]:
master_df = master_df.drop_duplicates()

In [35]:
master_df.shape

(175688, 8)

In [36]:
#Blank Bill No handle karo
master_df["Bill No"].isna().sum()

np.int64(18)

In [37]:
master_df = master_df.dropna(
    subset=["Bill No"]
)

In [38]:
#Final Missing Values Check
master_df.isna().sum()

Customer Name          0
Product Category       0
Bill No                0
Bill Date              0
Product Name           0
R/I                    0
Qty                    0
Rate                1085
dtype: int64

In [39]:
#Final Column Order
final_columns = [
    "Customer Name",
    "Product Category",
    "Bill No",
    "Bill Date",
    "Product Name",
    "R/I",
    "Qty",
    "Rate"
]

master_df = master_df[final_columns]

In [40]:
#Final Data Validation
master_df.shape

(175670, 8)

In [41]:
master_df.columns.tolist()

['Customer Name',
 'Product Category',
 'Bill No',
 'Bill Date',
 'Product Name',
 'R/I',
 'Qty',
 'Rate']

In [42]:
#Customer missing
master_df["Customer Name"].isna().sum()

np.int64(0)

In [43]:
#Category missing
master_df["Product Category"].isna().sum()

np.int64(0)

In [44]:
#Total rows
master_df[
    master_df["Product Name"].astype(str).str.lower() == "total"
].shape

(0, 8)

In [45]:
#Negative Qty
(master_df["Qty"] < 0).sum()

np.int64(25354)

In [46]:
#Final Excel Master File save karo
output_file = r"C:\Users\komal\Videos\Power BI\Pandas Project\Master_Sales_PowerBI_Final_Cleaned.xlsx"

master_df.to_excel(
    output_file,
    index=False,
    sheet_name="Master Data"
)

In [47]:
print("File saved:", output_file)

File saved: C:\Users\komal\Videos\Power BI\Pandas Project\Master_Sales_PowerBI_Final_Cleaned.xlsx
